# Analisis y predicción de Ventas Parte 3.

In [ ]:
# Primero conectamos con Google Drive (Si esque estamos trabajando en Google Colab).
from google.colab import drive
drive.mount('/content/drive')

In [32]:
# Importamos librerias.
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Cargar los Datos

In [33]:
# Carga el archivo retail_sales.csv en un DataFrame de Pandas.
path = "../data/retail_sales_dataset.csv"
df = pd.read_csv(path)
# Muestra las primeras 10 filas del DataFrame para confirmar que los datos se han cargado correctamente.
df.head(10)

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100
5,6,2023-04-25,CUST006,Female,45,Beauty,1,30,30
6,7,2023-03-13,CUST007,Male,46,Clothing,2,25,50
7,8,2023-02-22,CUST008,Male,30,Electronics,4,25,100
8,9,2023-12-13,CUST009,Male,63,Electronics,2,300,600
9,10,2023-10-07,CUST010,Female,52,Clothing,4,50,200


# Transformación de Datos

- Crea nuevas columnas: Basándonos en los datos existentes, crea nuevas columnas que sean útiles para el análisis. Por ejemplo, calcula el ingreso total por venta y normaliza las ventas.

In [34]:
# Nueva columna para normalizar Total Amount.
max_value = df['Total Amount'].max()
min_value = df['Total Amount'].min()
# Usamos una función lambda para normalizar el valor en base al mínimo y máximo de esa columna.
df['TA Normalized'] = df['Total Amount'].apply(lambda x: (x - min_value) / (max_value - min_value))

In [35]:
# Nueva columna para normalizar Quantity.
max_value = df['Quantity'].max()
min_value = df['Quantity'].min()
# Usamos una función lambda para normalizar el valor en base al mínimo y máximo de esa columna.
df['Q Normalized'] = df['Quantity'].apply(lambda x: (x - min_value) / (max_value - min_value))

In [36]:
# Observamos como quedo el dataframe.
df.head()

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount,TA Normalized,Q Normalized
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150,0.063291,0.666667
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000,0.493671,0.333333
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30,0.002532,0.000000
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500,0.240506,0.000000
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100,0.037975,0.333333


- Clasifica los datos: Crea una columna que clasifique las ventas en categorías significativas (e.g., ‘Alta’, ‘Media’, ‘Baja’).

In [37]:
# Primero observamos los tipos de datos existentes en la columna Quantity.
df["Quantity"].unique()

array([3, 2, 1, 4])

Si las cantidades compradas van de 1 a 4, podrimaos definirlo de la siguiente forma:
- 1: Baja.
- 2,3: Media.
- 4: Alta.
Generaremos una nueva columna llamada Tipo Venta con estos valores.

In [38]:
# Primero definimos una función para clasificar el tipo de venta.
def classify_sale(num: int)->str:
    if (num == 4):
        return "High"
    elif (num >= 2 and num <= 3):
        return "Mid"
    elif (num == 1):
        return "Low"
    else:
        return "Error"

In [39]:
# Usamos una función lambda para añadir la columna nueva aplicando nuestra función.
df['Type of Sale'] = df['Quantity'].apply(lambda x: classify_sale(x))
# Finalmente vemos como quedan los datos.
df.head()

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount,TA Normalized,Q Normalized,Type of Sale
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150,0.063291,0.666667,Mid
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000,0.493671,0.333333,Mid
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30,0.002532,0.000000,Low
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500,0.240506,0.000000,Low
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100,0.037975,0.333333,Mid


# Agrupación y Agregación

- Agrupación por múltiples columnas: Realiza agrupaciones por categorías como producto y tienda, producto y mes, etc.
- Aplicar funciones de agregación: Utiliza funciones como sum, mean, count, min, max, std, y var para obtener estadísticas descriptivas de cada grupo.

In [40]:
# Agrupo por producto y genero, y obtengo solamente el Total Amount Normalizado. 
# Aun si el ejercicio pide que utilice la "Tienda", estas no vienen incluidas en los datos.
df.groupby(["Product Category","Gender"])["TA Normalized"].describe()

count      mean       std  min       25%       50%  \
Product Category Gender                                                       
Beauty           Female  166.0  0.215586  0.272777  0.0  0.025316  0.063291   
                 Male    141.0  0.233989  0.300202  0.0  0.025316  0.048101   
Clothing         Female  174.0  0.223847  0.292160  0.0  0.019620  0.048101   
                 Male    177.0  0.199900  0.265375  0.0  0.017722  0.063291   
Electronics      Female  170.0  0.215890  0.277795  0.0  0.017722  0.063291   
                 Male    172.0  0.223344  0.297281  0.0  0.012658  0.048101   

                              75%  max  
Product Category Gender                 
Beauty           Female  0.443038  1.0  
                 Male    0.443038  1.0  
Clothing         Female  0.443038  1.0  
                 Male    0.291139  1.0  
Electronics      Female  0.443038  1.0  
                 Male    0.443038  1.0

Ya que agrupamos por productos y luego por genero, podemos obtener el siguiente analisis de este describe:
1. Los hombres gastan más en productos de la categoría Belleza que las mujeres.
2. Las mujeres gastan más en productos de la categoría Ropa que los hombres.
3. Los hombres gastan más en productos de la categoría Electronicos que las mujeres.

# Análisis Personalizado con apply

- Función personalizada: Aplica funciones personalizadas para realizar análisis específicos que no se pueden lograr con las funciones de agregación estándar.
- Ejemplo de uso avanzado: Calcula la desviación de cada venta respecto a la media de su grupo.

In [42]:
# La nueva columna será la desviación con respecto a la media que hay en TA Normalized.
# Obtenemos la media.
media_ta_normalized = df["TA Normalized"].mean()
print(f"Media de 'TA Normalized': {media_ta_normalized}")

Media de 'TA Normalized': 0.2182278481012658


In [43]:
# Calculamos la desviación de cada fila respecto a la media.
# Definimos una función para hacerlo.
def calculate_desv(num: float)->float:
    """
    Calcula la desviación de un valor 'TA Normalized' respecto a la media de 'TA Normalized'.
    """
    return num - media_ta_normalized

In [44]:
# Aplicamos la función a la columna 'TA Normalized' generando una nueva columna.
df["TA Desv"] = df["TA Normalized"].apply(lambda x: calculate_desv(x))
# Mostramos el dataframe actualizado.
df.head()

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount,TA Normalized,Q Normalized,Type of Sale,TA Desv
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150,0.063291,0.666667,Mid,-0.154937
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000,0.493671,0.333333,Mid,0.275443
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30,0.002532,0.000000,Low,-0.215696
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500,0.240506,0.000000,Low,0.022278
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100,0.037975,0.333333,Mid,-0.180253


De los primeros elementos que aparecen con head podemos observar desviaciones positivas y negativas, lo que nos indica que hay elementos que están por sobre la media, y otros que están por debajo de la media.

Datos que obtuvimos agrupando, añadiendo funciones y utilizando apply.